In [13]:
import requests
from pathlib import Path
import MDAnalysis as mda
from MDAnalysis.lib.util import NamedStream
from io import StringIO
import gemmi
import pymol
from glob import glob

In [14]:
uniprot_ids = {
               "CYP2D6": "P10635", 
               "CYP3A4": "P08684",
               "CYP1A2": "P05177",
               "CYP2C9": "P11712", 
              }


In [15]:
# User defined
common_co_crystals = ["GOL", "IMD", "SO4", "EDO", "PO4", "DMS", "CIT", 
                      "ZN", "NA", "K",  #ions
                     "MPD", "IPA", #solvents
                     "2CV", "CPS"] #detergents

exclude_co_crystals = " ".join([f"and not resname {c}" for c in common_co_crystals ])

In [16]:
#Code below will only do one ID at a time. Set interested KEY below.
target = "CYP2D6"

#### **If you already have input data to process, skip to proper section. 
Not all sections should be run. 

Run cells for your needs. 

The following options are available:

- Download and save raw pdb files from RCSB and save processed final files
- Download raw pdb text from RCSB and save processed final files
- Have existing pdb files that you are interested in processing, and save final files

## Query PDB IDs

Get PDB IDs from UNIPROT ID using RCSB PDB search API

In [17]:
def get_pdb_ids(uniprot_id, rows=1000):
    url = "https://search.rcsb.org/rcsbsearch/v2/query?json="
    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                "operator": "exact_match",
                "value": uniprot_id
            }
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {
                "start": 0,
                "rows": rows  # default 10, change if 1000 ids found 
            }
        }
    }
    response = requests.post(url, json=query)
    response.raise_for_status()
    result = response.json()
    pdb_ids = [entry["identifier"] for entry in result["result_set"]]
    if len(pdb_ids) < rows:
        print(f"Found {len(pdb_ids)} PDB IDs.")
    else:
        print(f"Found {len(pdb_ids)} PDB IDs. Consider changing rows to greater than {rows}.")
    return pdb_ids

In [18]:
pdb_ids = get_pdb_ids(uniprot_ids[target])

Found 14 PDB IDs.


If using IDs, this downloads PDBs using the RCSB API

In [19]:
def get_rcsb_url(pdb_id, fmt="pdb"):
    url = f"https://files.rcsb.org/download/{pdb_id}.{fmt}"
    return requests.get(url)

def write_file(text, file_path):
    with open(file_path, "w") as f:
        f.write(text)

def convert_cif_to_pdb_gemmi(cif_text):
    """Convert mmCIF text to PDB string using gemmi."""
    doc = gemmi.cif.read_string(cif_text)
    structure = gemmi.make_structure_from_block(doc.sole_block())
    return structure.make_pdb_string()


In [20]:
def get_rcsb_pdb(pdb_id, outdir=".", download_initial=False):
    """
    Download a PDB or mmCIF for the given PDB ID.
    Returns the final PDB content as a string. 
    The initial data file is saved, if specified. 
    """
    if download_initial:
        outdir = Path(outdir)
        outdir.mkdir(parents=True, exist_ok=True)
        pdb_path = outdir / f"{pdb_id}_initial.pdb"

    # First try direct PDB download
    pdb_response = get_rcsb_url(pdb_id, fmt="pdb")
    if pdb_response.status_code == 200:
        print(f"Downloading {pdb_id} from RCSB...")
        pdb_text = pdb_response.text
        if download_initial:
            print(f"Saving {pdb_id} from RCSB...")
            write_file(pdb_text, pdb_path)
        return pdb_text

    # Fallback to CIF
    print(f"PDB for {pdb_id} not found. Checking for mmCIF...")
    cif_response = get_rcsb_url(pdb_id, fmt="cif")
    if cif_response.status_code == 200:
        print(f"mmCIF found. Converting to PDB...")
        pdb_text = convert_cif_to_pdb_gemmi(cif_response.text)
        if download_initial:
            write_file(pdb_text, pdb_path)
        return pdb_text

    raise ValueError(f"Neither PDB nor CIF available for {pdb_id}.")

def get_pdb_path(pdb_dir, pdb_id):
    return glob(f"{pdb_dir}//*{pdb_id}*.pdb")[0]



In [21]:
def process_pdb(pdb_id, 
                pdb_text="", 
                outdir=".", 
                exclude_co_crystals="", 
                input_path=None):
    """
    Process the PDB text to remove common co-crystals and
    save the final processed PDB.
    """
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    final_path = outdir / f"{pdb_id}.pdb"

    if input_path:
        pdb = get_pdb_path(input_path, pdb_id)
        print(pdb)
        u = mda.Universe(pdb)
    else:
        u = mda.Universe(NamedStream(StringIO(pdb_text), f"{pdb_id}.pdb"))
    protein_A = u.select_atoms("protein and chainid A")
    others = u.select_atoms(f"chainid A and not protein and not water {exclude_co_crystals}")

    #Currently removing HEM dependency as it is only necessary in CYP, and doesn't include occurence of HEC
    # if 'HEM' not in set(others.resnames):
    #     print(f"Skipping {pdb_id}: HEM group not found.")
    #     return None
        
    #assert 'HEM' in set(others.resnames), "HEM group not found in structure."

    combined = protein_A + others
    lig = combined.select_atoms(f"chainid A and not protein and not resname HEM")
    
    if len(lig) > 0:
        lig.residues.resnames = ["LIG"] * len(lig.residues)
        combined = combined + lig

    final_lig_set = set(combined.select_atoms("chainid A and not protein and not water").resnames)
    #assert final_lig_set == {"HEM"} or final_lig_set == {"LIG", "HEM"}

    combined.write(final_path)
    return final_path

## Download Initial Files from RCSB and Save Processed Files

In [22]:
input_dir = f"{target}/raw_pdb"
final_dir = f"{target}/final"

In [23]:
for _id in pdb_ids:
    _id = _id.lower()
    print(f"\nProcessing: {_id}")
    pdb_text = get_rcsb_pdb(_id, outdir=input_dir, download_initial=True)
    try:
        processed_path = process_pdb(_id, pdb_text, outdir=final_dir, exclude_co_crystals=exclude_co_crystals)
        print(f"Saved processed PDB: {processed_path}")
    except Exception as e:
        print(f"Failed to process {_id}: {e}")
    


Processing: 2f9q
Saving 2f9q from RCSB...
Saved processed PDB: CYP2D6/final/2f9q.pdb

Processing: 3qm4
Saving 3qm4 from RCSB...
Saved processed PDB: CYP2D6/final/3qm4.pdb

Processing: 3tbg
Saving 3tbg from RCSB...
Saved processed PDB: CYP2D6/final/3tbg.pdb

Processing: 3tda
Saving 3tda from RCSB...
Saved processed PDB: CYP2D6/final/3tda.pdb

Processing: 4wnt
Saving 4wnt from RCSB...
Saved processed PDB: CYP2D6/final/4wnt.pdb

Processing: 4wnu
Saving 4wnu from RCSB...
Saved processed PDB: CYP2D6/final/4wnu.pdb

Processing: 4wnv
Saving 4wnv from RCSB...
Saved processed PDB: CYP2D6/final/4wnv.pdb

Processing: 4wnw
Saving 4wnw from RCSB...
Saved processed PDB: CYP2D6/final/4wnw.pdb

Processing: 4xry
Saving 4xry from RCSB...
Saved processed PDB: CYP2D6/final/4xry.pdb

Processing: 4xrz
Saving 4xrz from RCSB...
Saved processed PDB: CYP2D6/final/4xrz.pdb

Processing: 5tft
Saving 5tft from RCSB...
Saved processed PDB: CYP2D6/final/5tft.pdb

Processing: 5tfu
Saving 5tfu from RCSB...
Saved proce

## Only Grab RCSB PDB text and Save Processed Files

In [ ]:
final_dir = f"{target}/final"

In [ ]:
for _id in pdb_ids:
    _id = _id.lower()
    print(f"\nProcessing: {_id}")
    pdb_text = get_rcsb_pdb(_id)
    try:
        processed_path = process_pdb(_id, pdb_text, outdir=final_dir, exclude_co_crystals=exclude_co_crystals)
        print(f"Saved processed PDB: {processed_path}")
    except Exception as e:
        print(f"Failed to process {_id}: {e}")

## Process from existing PDB

In [ ]:
pdb_dir_path = 'PATH/TO/PDB/FILES'

for _id in pdb_ids:
    _id = _id.lower()
    try:
        processed_path = process_pdb(_id, pdb_text, outdir=final_dir, exclude_co_crystals=exclude_co_crystals, input_path=pdb_dir_path)
        print(f"Saved processed PDB: {processed_path}")
    except Exception as e:
        print(f"Failed to process {_id}: {e}")